# 38 — Transformer Complexity and Failure Analysis

**Master syllabus coverage:** V2 5.15–5.16

## Why this matters

A correct model must also fit memory, meet latency goals, and fail loudly when invariants break. Complexity estimates and targeted tests turn debugging from guesswork into engineering.

## Learning objectives

- Estimate linear-layer and attention compute scaling.
- Estimate materialized attention memory against context length.
- Design targeted shape, causality, finiteness, state, and resume tests.
- Interpret attention weights without treating them as complete explanations.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Separate parameter, compute, and activation scaling

For batch $B$, length $T$, width $C$, heads $H$, layers $L$:

- QKV + output projections: roughly $4BTC^2$ multiply-adds per layer;
- attention score and value products: roughly $2BT^2C$ per layer;
- classic 4C MLP: roughly $8BTC^2$ per layer;
- attention matrix storage: $BHT^2$ values per layer when materialized.

Long context makes the quadratic term dominant; wide models make projection/MLP terms dominant.


In [ ]:
import math
import time
import torch

def rough_layer_work(batch, time_steps, width):
    linear = 12 * batch * time_steps * width**2
    attention = 2 * batch * time_steps**2 * width
    return linear, attention

for T in (128, 512, 2048, 8192):
    linear, attention = rough_layer_work(1, T, 768)
    print(f"T={T:5}: linear={linear/1e9:8.2f}G, attention={attention/1e9:8.2f}G, ratio={attention/linear:.3f}")


## 2. Attention memory grows quadratically

Optimized exact attention such as FlashAttention reduces intermediate memory traffic and
avoids materializing the full matrix, but does not change the mathematical dense-attention
pair count. Fused kernels improve constants and memory behavior; sparse/local schemes
change which pairs exist and therefore the computation.


In [ ]:
def attention_matrix_gib(batch, heads, context, bytes_per_value=2):
    return batch * heads * context * context * bytes_per_value / 2**30

for T in (512, 2048, 8192, 32768):
    print(f"T={T:5}: one 12-head fp16 score/prob matrix = {attention_matrix_gib(1,12,T):.3f} GiB")


## 3. Build failure tests before trusting training

High-value invariants include shape/dtype/device checks, finite logits/loss/gradients,
probability-row sums, no future influence, weight-tie identity, deterministic eval,
overfit-one-batch, save/load equivalence, and exact-resume tests. A decreasing full training
curve is too indirect to localize most failures.


In [ ]:
def causal_leak_test(model, prefix, suffix_a, suffix_b, atol=1e-5):
    model.eval()
    with torch.inference_mode():
        first = model(torch.cat((prefix, suffix_a), dim=1))[0]
        second = model(torch.cat((prefix, suffix_b), dim=1))[0]
    torch.testing.assert_close(first[:, :prefix.shape[1]], second[:, :prefix.shape[1]],
                               atol=atol, rtol=atol)

# Demonstrate the invariant directly on a correct causal weight matrix.
T = 6
allowed = torch.ones(T, T, dtype=torch.bool).tril()
logits = torch.randn(2, 3, T, T).masked_fill(~allowed, float("-inf"))
probabilities = torch.softmax(logits, dim=-1)
assert not probabilities.triu(diagonal=1).any()
torch.testing.assert_close(probabilities.sum(dim=-1), torch.ones(2, 3, T))


## 4. Attention weights are evidence, not explanations

A large weight says a query used a value strongly in that head at that layer. The value
vector, output projection, residual stream, later layers, and nonlinear transformations
determine effect. Attention can help debug masks and patterns, but it is not automatically
a faithful explanation of a final prediction.


In [ ]:
# Same attention weights, different values → different outputs.
weights = torch.tensor([[0.1, 0.9]])
values_a = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
values_b = torch.tensor([[100.0, 0.0], [0.0, -1.0]])
print("same weights, output A:", weights @ values_a)
print("same weights, output B:", weights @ values_b)
assert not torch.allclose(weights @ values_a, weights @ values_b)


## 5. Benchmarking needs a question

Report hardware/software, dtype, shapes, warm-up, repeats, synchronization, memory metric,
and whether backward is included. Compare outputs within a precision-appropriate tolerance.
Throughput, latency, and peak memory are different objectives; batch size can improve one
while harming another.


In [ ]:
# A CPU-only, small semantic benchmark example—not a universal performance claim.
q = torch.randn(1, 4, 256, 32)
k = torch.randn_like(q)
v = torch.randn_like(q)
for _ in range(3):
    _ = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
start = time.perf_counter()
repeats = 10
for _ in range(repeats):
    result = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
elapsed = time.perf_counter() - start
print(f"shape={tuple(q.shape)}, dtype={q.dtype}, mean latency={elapsed/repeats*1e3:.3f} ms")
assert torch.isfinite(result).all()


## Failure modes to recognize

- **Quadratic term ignored:** a context increase exceeds compute or memory budget.
- **Optimized-kernel mythology:** fused exact attention is claimed to change dense asymptotic pair count.
- **Loss-only debugging:** silent leakage or state bugs remain unlocalized.
- **Benchmark without equivalence:** faster outputs solve a different or numerically broken problem.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Find the context length where attention work equals linear work for width C under the rough formulas.
2. Write a failure-test matrix mapping ten symptoms to the smallest invariant test.
3. Design a benchmark report comparing manual attention and SDPA on your available device.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can budget a configuration, define its performance question, and select a minimal test for each major Transformer failure class.

**Next:** 39 — Tiny decoder readiness capstone.
